# VM Behavioural Clustering

Load and preserve the source data before feature engineering or cleaning.

In [7]:
%pip install matplotlib seaborn scikit-learn

  Using cached contourpy-1.3.3-cp313-cp313-win_amd64.whl.metadata (5.5 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
   ---------------------------------------- 0.0/9.3 MB ? eta -:--:--
   ---------------------------------------  9.2/9.3 MB 48.3 MB/s eta 0:00:01
   ---------------------------------------- 9.3/9.3 MB 18.8 MB/s  0:00:00
   ---------------------------------------- 0.0/8.2 MB ? eta -:--:--
   ---------------------------------------- 8.2/8.2 MB 36.5 MB/s  0:00:00
Using cached contourpy-1.3.3-cp313-cp313-win_amd64.whl (226 kB)
Using cached cycler-0.12.1-py3-none-any.whl (8.3 kB)
   ---------------------------------------- 0.0/2.3 MB ? eta -:--:--
   ---------------------------------------- 2.3/2.3 MB 37.3 MB/s  0:00:00
   ---------------------------------------- 0.0/7.2 MB ? eta -:--:--
   ---------------------------------------  7.1/7.2 MB 82.9 MB/s eta 0:00:01
   ---------------------------------------  7.1/7.2 MB 82.9 MB/s eta 0:00:01
   ------------

In [8]:
# Reproducible notebook setup
from pathlib import Path
import random
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 42
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', context='notebook')
pd.set_option('display.max_columns', 100)

Matplotlib is building the font cache; this may take a moment.


In [9]:
# Load source tables
DATA_DIR = Path.cwd()

vm_info = pd.read_csv(DATA_DIR / 'vm_info.csv')
vm_utilization = pd.read_csv(DATA_DIR / 'vm_utilization_timeseries.csv')
cpu_type_definitions = pd.read_csv(DATA_DIR / 'cpu_type_definitions.csv')
memory_type_definitions = pd.read_csv(DATA_DIR / 'memory_type_definitions.csv')

# Immutable-in-practice backups: retain these unchanged throughout cleaning and analysis.
vm_info_raw = vm_info.copy(deep=True)
vm_utilization_raw = vm_utilization.copy(deep=True)

print(f'VM metadata: {vm_info.shape[0]:,} rows × {vm_info.shape[1]} columns')
print(f'Utilization readings: {vm_utilization.shape[0]:,} rows × {vm_utilization.shape[1]} columns')

VM metadata: 1,500 rows × 8 columns
Utilization readings: 765,189 rows × 8 columns


In [10]:
# Standardize the two same-structured size lookup tables and stack them into one reference table.
cpu_reference = (
    cpu_type_definitions
    .rename(columns={'core_count_bucket': 'size_bucket'})
    .assign(resource_type='cpu')
)
memory_reference = (
    memory_type_definitions
    .rename(columns={'memory_bucket': 'size_bucket'})
    .assign(resource_type='memory')
)

size_bucket_reference = pd.concat(
    [cpu_reference, memory_reference],
    ignore_index=True,
)

size_bucket_reference

,size_bucket,bucket_category,min_value,max_value,representative_value,resource_type
0,small_2_4,cores,2,4,2,cpu
1,medium_4_8,cores,4,8,4,cpu
2,large_8_16,cores,8,16,8,cpu
3,xlarge_16_32,cores,16,32,16,cpu
4,xxlarge_32_64,cores,32,64,32,cpu
5,small_4_8,memory_gb,4,8,4,memory
6,medium_8_16,memory_gb,8,16,8,memory
7,large_16_32,memory_gb,16,32,16,memory
8,xlarge_32_64,memory_gb,32,64,32,memory
9,xxlarge_64_128,memory_gb,64,128,64,memory
